# Benchmark Overview

Inspect MR-GSM8K, Big-Bench Mistake and ProcessBench to verify that we can use them for the thesis.

In [2]:
%load_ext autoreload
%autoreload 2

print("Loaded extensions")

Loaded extensions


In [ ]:
from pathlib import Path

from datasets import load_dataset
from dotenv import load_dotenv
from huggingface_hub import login
from os import environ
import ast
import pandas as pd

EXPERIMENTS = Path.cwd().parent

In [ ]:
load_dotenv(EXPERIMENTS.parent / 'secret.env')
assert environ.get("HF_TOKEN", "").startswith("hf_"), "No HF_TOKEN configured!"
login()

## MR-GSM8K

In [5]:
mr_gsm8k_dict = load_dataset("Randolphzeng/Mr-GSM8K")
mr_gsm8k_dict

DatasetDict({
    test: Dataset({
        features: ['Unnamed: 0', 'uuid', 'question', 'ground_truth_solution', 'ground_truth_answer', 'model_output_steps', 'model_output_answer_correctness', 'model_output_solution_correctness', 'model_output_solution_first_error_step', 'model_output_solution_first_error_reason', 'question_type'],
        num_rows: 2999
    })
})

In [6]:
# .with_format("pandas") is a scam
#  always use .to_pandas()
mr_gsm8k = mr_gsm8k_dict["test"].to_pandas().set_index("uuid").drop("Unnamed: 0", axis='columns').astype({"model_output_solution_first_error_step": "Int64"})
assert mr_gsm8k.index.is_unique
mr_gsm8k.model_output_steps = mr_gsm8k.model_output_steps.apply(ast.literal_eval)
mr_gsm8k.head()

,question,ground_truth_solution,ground_truth_answer,model_output_steps,model_output_answer_correctness,model_output_solution_correctness,model_output_solution_first_error_step,model_output_solution_first_error_reason,question_type
uuid,,,,,,,,,
f25b796f-bc99-49c7-a6cd-ac5bb98412ce,There are 6 girls in the park. If there are tw...,There are 6 girls x 2 boys/girl = <<6*2=12>>12...,18,[Step 1: There are twice the number of boys i...,correct,correct,<NA>,NaN,original
d1cea8ad-2dc4-4ad8-8bfe-810140f59b09,A glass of milk is 8 ounces of milk. John dri...,He drank 8*2=<<8*2=16>>16 ounces of milk\nSo h...,48,[Step 1: He drank 2*8=<<2*8=16>>16 ounces of ...,correct,correct,<NA>,NaN,original
b27daa8c-a94d-4240-99e8-24f82af5ff1c,Tom gets 4 car washes a month. If each car wa...,He gets 4*12=<<4*12=48>>48 car washes a year\n...,720,[Step 1: He pays 4*15=$<<4*15=60>>60 per mont...,correct,correct,<NA>,NaN,original
77511c73-071d-448b-8c1e-1210131189b9,"Carmen has $100, Samantha has $25 more than Ca...",Samantha has $100+$25 = $<<100+25=125>>125.\nD...,400,[Step 1: Carmen has $100. Samantha has $25 mo...,correct,correct,<NA>,NaN,original
c6b8499f-d3ab-4769-bd41-92b650591832,An electronics seller bought 5 phones for $700...,The seller bought the 5 phones for $700 * 5 = ...,500,[Step 1: The seller bought 5 phones for 5*700...,correct,correct,<NA>,NaN,original


In [7]:
mr_gsm8k["model_output_solution_correctness"].value_counts()

model_output_solution_correctness
wrong      1571
correct    1428
Name: count, dtype: int64

In [8]:
mr_gsm8k.query("model_output_solution_correctness == 'wrong'").head()

,question,ground_truth_solution,ground_truth_answer,model_output_steps,model_output_answer_correctness,model_output_solution_correctness,model_output_solution_first_error_step,model_output_solution_first_error_reason,question_type
uuid,,,,,,,,,
2b038c65-0914-45ab-9fa0-e6e59b869705,Mico and Marco wanted to get to know each othe...,There will be an additional 10 (for Mico) + 10...,40,"[Step 1: If the sum of their ages is 20, then...",correct,wrong,1,The solution incorrectly assumes the age of bo...,original
e24fba0b-630c-41d5-a50d-975d877df908,James decided to walk to the store. When he g...,He walked 4/2=<<4/2=2>>2 miles there before tu...,2,[Step 1: He had to walk to the store which is...,correct,wrong,1,The solution misunderstands the question is on...,original
aa940a1b-c211-4ca8-9c64-92564a0dd712,A farmer is buying feed for his horses. He buy...,The farmer is buying 2 * 1 = <<2*1=2>>2 pounds...,2,[Step 1: The farmer buys 2 boxes of sugar cub...,wrong,wrong,1,the solution incorrectly assumed a non-exist i...,original
bd29bebd-0d3a-4700-90be-23c0c4c4e96c,"Bob is 75 miles ahead of Tom, driving 55 miles...",Tom is driving at 70 miles per hour to Bob's 5...,5,[Step 1: We know that the distance between bo...,wrong,wrong,1,The solution assumes Bob is static and did not...,original
8eb87f9c-a87f-480d-9535-9595eb768e52,Michael is traveling on the interstate at an a...,"There are 24 hours in each day, so he has been...",60,[Step 1: We know that distance = speed * time...,wrong,wrong,1,"The speed is with unit kilometers/hour, howeve...",original


In [9]:
q_types = mr_gsm8k.question_type.unique()
for q_type in q_types:
    sample = mr_gsm8k.query(f"model_output_solution_correctness == 'wrong' and question_type == '{q_type}'").iloc[0]
    print(q_type)
    print(sample.question)
    print("\n".join(sample.model_output_steps))
    print(f"Answer is {sample.model_output_answer_correctness}")
    print(f"Reasoning trace is {sample.model_output_solution_correctness}")
    print(f"First error occurs at step {sample.model_output_solution_first_error_step}")
    print()

original
Mico and Marco wanted to get to know each other. They realized that the sum of their ages is 20. What will be the sum of their ages in 10 years?
Step 1:  If the sum of their ages is 20, then each of them is 20/2 = <<20/2=10>>10 years old.
Step 2: In 10 years, Mico will be 10+10 = <<10+10=20>>20 years old.
Step 3: In 10 years, Marco will be 10+10 = <<10+10=20>>20 years old.
Step 4: So, the sum of their ages in 10 years will be 20+20 = <<20+20=40>>40.
Step 5: #### 40
Answer is correct
Reasoning trace is wrong
First error occurs at step 1

POT
If Chester eats 3 eggs a day for 30 days and then increases it to 5 eggs a day for 30 days, how many dozens of eggs will Chester need for 60 days?
Step 1: months_eating_3_eggs=2
Step 2: increased_eggs=5
Step 3: days_eating_5_eggs=30
Step 4: months_eating_5_eggs=6
Step 5: total_days=60
Step 6: total_months=6+2
Step 7: total_eggs_months=months_eating_3_eggs*30+months_eating_5_eggs*days_eating_5_eggs+total_months
Step 8: total_dozen_eggs=total

## Big-Bench Mistake

In [ ]:
BBM_TASK_TYPES = ["dyck_languages", "logical_deduction", "multistep_arithmetic", "tracking_shuffled_objects", "word_sorting"]
task_dfs = []
for task_type in BBM_TASK_TYPES:
    df = pd.read_json(EXPERIMENTS / 'BIG-Bench-Mistake' / f'{task_type}.jsonl', lines=True).astype({"mistake_index": "Int64"})
    df["task_type"] = task_type
    task_dfs.append(df)
bbm = pd.concat(task_dfs)
bbm.head()

In [11]:
for task_type in BBM_TASK_TYPES:
    sample = bbm.query(f"task_type == '{task_type}'").iloc[0]
    print(task_type)
    print(sample.input)
    print("\n".join(sample.steps))
    print()

dyck_languages
{ [ [ [ { [ ] } ] ]
We should process each input one by one and keep track of the stack configuration.
stack: empty
{ ; stack: {
[ ; stack: { [
[ ; stack: { [ [
[ ; stack: { [ [ [
{ ; stack: { [ [ [ {
[ ; stack: { [ [ [ { [
] ; stack: { [ [ [ {
} ; stack: { [ [ [
] ; stack: { [ [
] ; stack: { [
] ; stack: {
Now, we have reached the end. The final stack is "{".
We will need to pop out "{" one by one in that order.
So, we need "}". So the answer is }

logical_deduction
The following statements each describe a set of five objects arranged in a fixed order. The statements are logically consistent.
- In an antique car show, there are five vehicles: a truck, a station wagon, a motorcyle, a convertible, and a hatchback.
- The convertible is newer than the truck.
- The station wagon is newer than the hatchback.
- The convertible is older than the hatchback.
- The station wagon is the second-newest.
Q: Given the above statements, which of the following is correct?
Options:
(A) Th

## ProcessBench

In [12]:
process_bench_dict = load_dataset("Qwen/ProcessBench")
process_bench_dict

DatasetDict({
    gsm8k: Dataset({
        features: ['id', 'generator', 'problem', 'steps', 'final_answer_correct', 'label'],
        num_rows: 400
    })
    math: Dataset({
        features: ['id', 'generator', 'problem', 'steps', 'final_answer_correct', 'label'],
        num_rows: 1000
    })
    olympiadbench: Dataset({
        features: ['id', 'generator', 'problem', 'steps', 'final_answer_correct', 'label'],
        num_rows: 1000
    })
    omnimath: Dataset({
        features: ['id', 'generator', 'problem', 'steps', 'final_answer_correct', 'label'],
        num_rows: 1000
    })
})

In [13]:
for name, ds in process_bench_dict.items():
    print(name)
    sample = ds.with_format("pandas")[0].iloc[0]
    print(sample.problem)
    print()

gsm8k
Sue lives in a fun neighborhood.  One weekend, the neighbors decided to play a prank on Sue.  On Friday morning, the neighbors placed 18 pink plastic flamingos out on Sue's front yard.  On Saturday morning, the neighbors took back one third of the flamingos, painted them white, and put these newly painted white flamingos back out on Sue's front yard.  Then, on Sunday morning, they added another 18 pink plastic flamingos to the collection. At noon on Sunday, how many more pink plastic flamingos were out than white plastic flamingos?

math
Determine the remainder of 194 (mod 11).

olympiadbench
Find the smallest number $n$ such that there exist polynomials $f_{1}, f_{2}, \ldots, f_{n}$ with rational coefficients satisfying

$$
x^{2}+7=f_{1}(x)^{2}+f_{2}(x)^{2}+\cdots+f_{n}(x)^{2} .
$$

omnimath
Can one find 4004 positive integers such that the sum of any 2003 of them is not divisible by 2003?



In [17]:
process_bench_dict["gsm8k"].to_pandas().iloc[0].label

np.int64(1)